In [16]:
import pandas as pd
import numpy as np

In [17]:
# 1. Definice cesty a načtení dat
file_path = r'C:\Users\.......'
sitename = ''

# Načtení dat s automatickým převedením FLUXNET chybějících hodnot na np.nan
df_raw = pd.read_csv(file_path, na_values=['-9999', '-9999.0'], parse_dates=['TIMESTAMP'])

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\.......'

In [ ]:
# 2. Vektorizovaná extrakce a fyzikální konverze proměnných pro rsofun
# Využíváme primárně '_F' (gap-filled) verze pro zajištění spojitosti řady
df_forcing = pd.DataFrame({
    'sitename': sitename,
    'date': df_raw['TIMESTAMP'],
    'temp': df_raw['TA_F'],                  # Teplota vzduchu [°C]
    'vpd': df_raw['VPD_F'] * 100,            # Tlakový deficit (VPD): hPa -> Pa
    'patm': df_raw['PA_F'] * 1000,           # Atmosférický tlak: kPa -> Pa
    'ppfd': df_raw['SW_IN_F'] * 0.46 * 0.0864, # Dopadající radiace na fotosyntetický tok fotonů (PAR) [mol m-2 d-1]
    'rain': df_raw['P_F'],                   # Srážky [mm/den]
    'snow': 0.0,                             # rsofun explicitně vyžaduje parametr sněhu (zde dummy hodnota 0)
    'co2': df_raw['CO2_F']                   # Gap-filled koncentrace CO2 [ppm]
})

In [11]:
# 3. Kontrola spojitosti (rsofun havaruje, pokud ve forcingu narazí na NA)
missing_stats = df_forcing.isna().sum()
print("Počet chybějících hodnot v modelovém vstupu:")
print(missing_stats[missing_stats > 0])

# Odstranění nekompletních dnů (u kvalitních FLUXNET dat by to mělo být minimum řádků)
df_forcing_clean = df_forcing.dropna()

KeyError: 'w'

In [6]:
# 4. Export do formátu Parquet 
# Parquet je pro přenos mezi Py/R ideální, protože striktně zachovává datové typy (např. datetime)
output_path = f'rsofun_forcing_{sitename}.parquet'
df_forcing_clean.to_parquet(output_path, index=False)
print(f"\nForcing data uložena do: {output_path}")

KeyError: 'co2'